# Capstone Project 1 - Property Price Prediction

# Date of Submission: 01-05-2026 (Friday)

In [ ]:
# Importing the required libraries. 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly
import joblib
import pickle
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor 
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PowerTransformer, RobustScaler, OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score 

In [ ]:
# Importing the dataset. 
ppp = pd.read_csv(r"D:\AI_ML_DS Course_IIT Guwahati (Sept 2025-June 2026)\Capstone Projects\01_ML Project\House Price.csv")
ppp

In [ ]:
# Checking the first 5 rows of the dataset. 
ppp.head()

In [ ]:
# Checking the last 5 rows of the dataset. 
ppp.tail()

In [ ]:
# Checking the shape of the dataset. 
ppp.shape

In [ ]:
# Checking the size of the dataset.
ppp.size

In [ ]:
# Checking an overview of the dataset.
ppp.info()

In [ ]:
# Checking the unique values in each column of the dataset. 
ppp.nunique()

In [ ]:
# Checking the descriptive statistics of all the "numeric" columns in the dataset. 
ppp.describe()

In [ ]:
# Checking the missing values (NaN) in each column of the dataset. 
ppp.isnull().sum()

In [ ]:
# Checking the duplicate rows in the dataset. 
ppp[ppp.duplicated()]

##### From the output above, we can infer that there are 401 duplicate rows in the dataset. Therefore, we will drop these rows from our analysis. 

In [ ]:
# Dropping the duplicate rows from the dataset for our analysis. 
ppp.drop_duplicates(subset = None, keep = "first", inplace = True)

In [ ]:
# Checking the column names in the dataset. 
ppp.columns

In [ ]:
# Checking the distribution for all the "numeric" columns in the dataset using histograms. 
for col in ppp.columns:
    if ppp[col].dtype == "object":
        continue
    plt.figure(figsize = (6, 4))
    sns.histplot(ppp[col], kde = True, bins = 5)
    plt.title(f"Distribution of {col}")
    plt.show()

In [ ]:
# Checking the distribution for all the "numeric" columns in the dataset using boxplots. 
for col in ppp.columns:
    if ppp[col].dtype == "object":
        continue
    plt.figure(figsize = (6, 4))
    sns.boxplot(x = ppp[col], showfliers = True)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.show()

In [ ]:
# Checking the correlation for all the "numeric" columns in the dataset using a correlation matrix. 
correlation = ppp.drop(columns = {"POSTED_BY", "BHK_OR_RK", "ADDRESS"}).corr()
correlation

In [ ]:
# Visualizing the above correlation using a heatmap. 
plt.figure(figsize = (10, 10))
sns.heatmap(correlation, annot = True, cmap = "Reds")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# Checking for all outliers in all the "numeric" columns in the dataset using the IQR method. 
for col in ppp.columns:
    if ppp[col].dtype == "object":
        continue

    Q1 = ppp[col].quantile(0.25)
    Q3 = ppp[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = ppp[(ppp[col] < lower_bound) | (ppp[col] > upper_bound)]
    print(f"Outliers in {col}: {outliers.shape[0]}")

In [ ]:
# Printing Q1, Q3, IQR, lower_bound, and upper_bound for each "numeric" column in the dataset. 
for col in ppp.columns:
    if ppp[col].dtype == "object":
        continue
    
    Q1 = ppp[col].quantile(0.25)
    Q3 = ppp[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    print(f"{col}:\nQ1 = {Q1.round(2)}\nQ3 = {Q3.round(2)}\nIQR = {IQR.round(2)}\nlower_bound = {lower_bound.round(2)}\nupper_bound = {upper_bound.round(2)}")
    print("\n" + "-" * 30 + "\n")

In [ ]:
# Checking for the skewness in all the "numeric" columns of the dataset. 
skewness = ppp.drop(columns = {"POSTED_BY", "BHK_OR_RK", "ADDRESS"}).skew()
skewness

In [ ]:
# Initializing the PowerTransformer.
pt = PowerTransformer(method = "yeo-johnson")

In [ ]:
# Listing the highly skewed columns. 
skewed_cols = ["SQUARE_FT", "TARGET(PRICE_IN_LACS)"]

# Fitting and transforming.
ppp[skewed_cols] = pt.fit_transform(ppp[skewed_cols])

In [ ]:
# Listing all the categorical columns as consolidate.  
categorical_cols = ["POSTED_BY", "BHK_OR_RK"]

In [ ]:
# Performing One-Hot Encoding of the above "categorical" columns.
ppp_encoded = pd.get_dummies(ppp, columns = categorical_cols, drop_first = True)

In [ ]:
# Dropping columns that are not useful for math.
ppp_final = ppp_encoded.drop(columns = ["ADDRESS"]).astype(int)
ppp_final

In [ ]:
# Splitting the dataset into X and y. 
# X. 
X = ppp_final.drop(columns = ["TARGET(PRICE_IN_LACS)"])
X

In [ ]:
# y.
y = ppp_final["TARGET(PRICE_IN_LACS)"]
y

In [ ]:
# Splitting the dataset into training and testing sets. 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
# Feature scaling. 
# Creating an instance of StandardScaler. 
sc = StandardScaler()

In [ ]:
# Fitting and transforming the training dataset. 
X_train_scaled = sc.fit_transform(X_train)

In [ ]:
# Transforming the testing dataset.
X_test_scaled = sc.transform(X_test)

In [ ]:
# Defining various ML models. 
models = {
    "LR": LinearRegression(),
    "DT": DecisionTreeRegressor(ccp_alpha = 0.0, criterion = "friedman_mse", max_depth = 30, min_samples_leaf = 4, min_samples_split = 2),
    "RF": RandomForestRegressor(bootstrap = True, max_depth = 7, max_features = None, min_samples_split = 2, n_estimators = 100, random_state = 42),
    "SVM": SVR(),
    "KNN": KNeighborsRegressor(),
    "GB": GradientBoostingRegressor(learning_rate = 0.05, max_depth = 10, max_features = "log2", n_estimators = 100, subsample = 0.8)
}
models

In [ ]:
# Training the ML models defined above. 
results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)

# Making predictions for both training and testing datasets. 
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)

# Computing regression-based metrics. 
# Mean Squared Error (MSE) for both training and testing datasets. 
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)

# Root Mean Squared Error (RMSE) for both training and testing datasets. 
    train_rmse = np.sqrt(train_mse)
    test_rmse = np.sqrt(test_mse)

# R2 score for both training and testing datasets. 
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

# Checking for the overfitting. 
    overfitting = "Y" if (train_r2 - test_r2 > 0.1) else "N"

# Updating the empty list results = [], with the results. 
    results.append([name, train_mse, test_mse, train_rmse, test_rmse, train_r2, test_r2, overfitting])

In [ ]:
# Creating a DataFrame to compare the performance acorss all the models. 
results_df = pd.DataFrame(results, columns = ["Model", "Train MSE", "Test MSE", "Train RMSE", "Test RMSE", "Train R2", "Test R2", "Overfitting (Y/N)"])
print(results_df)

In [ ]:
# Hyperparameter tuning on DT, RF, and GB. 
# DT.
DT_param_grid = {
    "criterion": ["squared_error", "friedman_mse"],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "ccp_alpha": [0.0, 0.01, 0.1]  
}

RF_param_grid = {
    "n_estimators": [25, 50 ,100], 
    "max_depth": [3, 5, 7],
    "min_samples_split": [2, 5, 10],
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False]
}

GB_param_grid = {
    "n_estimators": [25, 50, 100],       
    "learning_rate": [0.01, 0.05, 0.1],   
    "max_depth": [10, 20, None],               
    "subsample": [0.8, 1.0],              
    "max_features": ["sqrt", "log2", None]      
}

In [ ]:
# Hyperparameter tuning for DecisionTreeRegressor using GridSearchCV().
DT_grid_search = GridSearchCV(estimator = DecisionTreeRegressor(), param_grid = DT_param_grid, cv = 5, scoring = "r2", n_jobs = -1)
DT_grid_search.fit(X_train_scaled, y_train)
print(f"Best Parameters for DecisionTreeRegressor: {DT_grid_search.best_params_}")

In [ ]:
# Hyperparameter tuning for RandomForestRegressor using GridSearchCV().
RF_grid_search = GridSearchCV(estimator = RandomForestRegressor(random_state = 42), param_grid = RF_param_grid, cv = 5, scoring = "r2", n_jobs = -1)
RF_grid_search.fit(X_train_scaled, y_train)
print(f"Best Parameters for RandomForestRegressor: {RF_grid_search.best_params_}")

In [ ]:
# Hyperparameter tuning for GradientBoostingRegressor using GridSearchCV().
GB_grid_search = GridSearchCV(estimator = GradientBoostingRegressor(), param_grid = GB_param_grid, cv = 5, scoring = "r2", n_jobs = -1)
GB_grid_search.fit(X_train_scaled, y_train)
print(f"Best Parameters for GradientBoostingRegressor: {GB_grid_search.best_params_}")

In [ ]:
# Storing the best model, i.e., RF.
best_rf_model = RF_grid_search.best_estimator_

In [43]:
# Saving the model. 
joblib.dump(best_rf_model, "Property_Price_Prediction.pkl")

['Property_Price_Prediction.pkl']

In [ ]:
# Saving the scaler. 
joblib.dump(sc, "Scaler.pkl")

In [ ]:
# Saving the column names. 
model_columns = list(X_train.columns)
joblib.dump(model_columns, "Model_columns.pkl")